In [ ]:
import pandas as pd
import numpy as np

# Hydrogen assumptions
ETA_H2 = 0.65        # efficiency
HHV_H2 = 39.4        # kWh/kg
H2_PRICE = 5.0       # €/kg
ELECTROLYZER_CAPEX = 800  # €/kW
OPEX_RATE = 0.03     # 3% of CAPEX

YEARS = 25
DISCOUNT_RATE = 0.0337
PRICE_GROWTH = 0.0449

def surplus_to_h2(surplus_kwh: pd.Series, max_power_kw: float) -> pd.DataFrame:
    usable_kwh = np.minimum(surplus_kwh, max_power_kw)  # <—— 新约束
    h2_energy = usable_kwh * ETA_H2
    h2_mass = h2_energy / HHV_H2
    return pd.DataFrame({
        "surplus_kwh": surplus_kwh,
        "usable_kwh": usable_kwh,
        "h2_energy_kwh": h2_energy,
        "h2_mass_kg": h2_mass
    })


def npv_h2(h2_mass_annual: float, max_power_kw: float) -> tuple[float, list[float]]:
    """NPV analysis for hydrogen plant."""
    capex = ELECTROLYZER_CAPEX * max_power_kw
    opex = capex * OPEX_RATE

    annual_revenue = h2_mass_annual * H2_PRICE

    cashflows = [-capex]
    npv = -capex
    for y in range(1, YEARS+1):
        benefit_y = annual_revenue * ((1 + PRICE_GROWTH) ** y) - opex
        disc = benefit_y / ((1 + DISCOUNT_RATE) ** y)
        cashflows.append(disc)
        npv += disc
    return npv, cashflows

def payback_year(cashflows: list[float]) -> float | None:
    cum = 0.0
    for y, cf in enumerate(cashflows):
        cum += cf
        if cum >= 0:
            return float(y)
    return None

# ===============================
# Main
# ===============================


In [7]:

# Load your offset results
df = pd.read_csv(r"./offset_results.csv")

# Use cooperative surplus (KL+PS)
surplus = df["KL+PS_surplus"]

# Convert surplus to hydrogen
h2_df = surplus_to_h2(surplus)

# Annual production
h2_mass_annual = h2_df["h2_mass_kg"].sum()
max_power_kw = surplus.max()  # electrolyzer size ~ peak surplus

print(f"Annual H2 production: {h2_mass_annual:.2f} kg")
print(f"Electrolyzer size needed: {max_power_kw:.1f} kW")

# NPV analysis
npv_val, cashflows = npv_h2(h2_mass_annual, max_power_kw)
payback = payback_year(cashflows)

print(f"NPV (25y): €{npv_val:,.2f}, Payback: {payback} years")

# Save results
summary = pd.DataFrame([{
    "annual_h2_kg": h2_mass_annual,
    "electrolyzer_kw": max_power_kw,
    "npv": npv_val,
    "payback_year": payback
}])
summary.to_csv("hydrogen_summary.csv", index=False)

# Also save hourly H2 production
# h2_df.to_csv("hydrogen_hourly.csv", index=False)
print("✅ Saved hydrogen_summary.csv and hydrogen_hourly.csv")


Annual H2 production: 35532506.70 kg
Electrolyzer size needed: 23574014.9 kW
NPV (25y): €-23,192,072,729.22, Payback: None years
✅ Saved hydrogen_summary.csv and hydrogen_hourly.csv


In [ ]:
# Load your offset results
df = pd.read_csv(r"./offset_results.csv")

# Use cooperative surplus (KL+PS)
surplus = df["PS_surplus"]

# Convert surplus to hydrogen
h2_df = surplus_to_h2(surplus)

# Annual production
h2_mass_annual = h2_df["h2_mass_kg"].sum()
max_power_kw = surplus.max()  # electrolyzer size ~ peak surplus

print(f"Annual H2 production: {h2_mass_annual:.2f} kg")
print(f"Electrolyzer size needed: {max_power_kw:.1f} kW")

# NPV analysis
npv_val, cashflows = npv_h2(h2_mass_annual, max_power_kw)
payback = payback_year(cashflows)

print(f"NPV (25y): €{npv_val:,.2f}, Payback: {payback} years")

# Save results
summary = pd.DataFrame([{
    "annual_h2_kg": h2_mass_annual,
    "electrolyzer_kw": max_power_kw,
    "npv": npv_val,
    "payback_year": payback
}])
summary.to_csv("hydrogen_summary.csv", index=False)

# Also save hourly H2 production
h2_df.to_csv("PS_hydrogen_hourly.csv", index=False)
print("✅ Saved PS_hydrogen_summary.csv and hydrogen_hourly.csv")


Annual H2 production: 11212007.57 kg
Electrolyzer size needed: 8717068.0 kW
NPV (25y): €-8,853,772,408.83, Payback: None years
✅ Saved hydrogen_summary.csv and hydrogen_hourly.csv


In [10]:
import pandas as pd
import numpy as np
from typing import Tuple, List, Optional, Dict

# -------------------------------
# Parameters (keep as you prefer)
# -------------------------------
ETA_H2 = 0.65               # electrolysis efficiency (electricity -> H2 energy)
HHV_H2 = 39.4               # kWh/kg H2 (higher heating value)
H2_PRICE = 5.0              # €/kg revenue (or avoided cost)
ELECTROLYZER_CAPEX = 800    # €/kW CAPEX
OPEX_RATE = 0.03            # annual OPEX as % of CAPEX

YEARS = 25
DISCOUNT_RATE = 0.0337
PRICE_GROWTH = 0.0449       # annual H2 price growth (nominal)

# Use the Pirmasens-Winzeln electrolyser rating: 1.8 MW = 1800 kW
MAX_POWER_KW = 1800.0


# -------------------------------
# Core functions
# -------------------------------
def surplus_to_h2(surplus_kwh: pd.Series, max_power_kw: float) -> pd.DataFrame:
    """
    Convert hourly surplus electricity (kWh) into hydrogen production,
    enforcing an electrolyser power cap (kW = kWh per hour).
    """
    # Ensure numeric and non-negative
    s = pd.to_numeric(surplus_kwh, errors="coerce").fillna(0.0).clip(lower=0.0)

    # Cap per-hour usable electricity by electrolyser power
    usable_kwh = np.minimum(s.values, max_power_kw)
    curtailed_kwh = s.values - usable_kwh

    h2_energy = usable_kwh * ETA_H2
    h2_mass = h2_energy / HHV_H2

    return pd.DataFrame(
        {
            "surplus_kwh": s.values,
            "usable_kwh": usable_kwh,
            "curtailed_kwh": curtailed_kwh,
            "h2_energy_kwh": h2_energy,
            "h2_mass_kg": h2_mass,
        },
        index=s.index,
    )


def summarize_h2(df: pd.DataFrame, max_power_kw: float) -> Dict[str, float]:
    """Summarize annual (or period) totals and equivalent full-load hours."""
    total_surplus = float(df["surplus_kwh"].sum())
    total_usable = float(df["usable_kwh"].sum())
    total_curtailed = float(df["curtailed_kwh"].sum())
    total_h2_energy = float(df["h2_energy_kwh"].sum())
    total_h2_mass = float(df["h2_mass_kg"].sum())
    flh = total_usable / max_power_kw if max_power_kw > 0 else 0.0
    return {
        "total_surplus_kwh": total_surplus,
        "total_usable_kwh": total_usable,
        "total_curtailed_kwh": total_curtailed,
        "curtailment_ratio": (total_curtailed / total_surplus) if total_surplus > 0 else 0.0,
        "total_h2_energy_kwh": total_h2_energy,
        "total_h2_mass_kg": total_h2_mass,
        "full_load_hours": flh,
    }


def npv_h2(h2_mass_annual: float, max_power_kw: float) -> Tuple[float, List[float]]:
    """NPV with CAPEX at t=0 and yearly (H2 revenue – OPEX) thereafter."""
    capex = ELECTROLYZER_CAPEX * max_power_kw
    opex = capex * OPEX_RATE
    annual_revenue = h2_mass_annual * H2_PRICE

    cashflows = [-capex]
    npv = -capex
    for y in range(1, YEARS + 1):
        benefit_y = annual_revenue * ((1 + PRICE_GROWTH) ** y) - opex
        discounted = benefit_y / ((1 + DISCOUNT_RATE) ** y)
        cashflows.append(discounted)
        npv += discounted
    return npv, cashflows


def payback_year(cashflows: List[float]) -> Optional[float]:
    """Discounted payback year (first year cumulative discounted CF >= 0)."""
    cum = 0.0
    for y, cf in enumerate(cashflows):
        cum += cf
        if cum >= 0:
            return float(y)
    return None


def lcoh_simple(h2_mass_annual: float, max_power_kw: float,
                discount_rate: float = DISCOUNT_RATE, years: int = YEARS) -> Optional[float]:
    """
    Simple LCOH (€/kg): annualized CAPEX + OPEX, divided by annual H2 mass.
    Assumes surplus electricity has no explicit cost; add it if needed.
    """
    if h2_mass_annual <= 0:
        return None
    capex = ELECTROLYZER_CAPEX * max_power_kw
    opex = capex * OPEX_RATE
    r, n = discount_rate, years
    crf = r * (1 + r) ** n / ((1 + r) ** n - 1)  # capital recovery factor
    annual_cost = capex * crf + opex
    return annual_cost / h2_mass_annual


# -------------------------------
# Read your CSV and run
# -------------------------------
# Read the file
df = pd.read_csv(r"./offset_results.csv")

# Pull the surplus column you mentioned
if "PS_surplus" not in df.columns:
    raise KeyError("Column 'PS_surplus' not found in offset_results.csv")

surplus_col = df["PS_surplus"]

# Build a DatetimeIndex if your CSV has a time column; otherwise synthesize one.
# Try to detect a timestamp-like column.
candidate_time_cols = [c for c in df.columns if "time" in c.lower() or "date" in c.lower()]
if candidate_time_cols:
    # Use the first candidate; parse to datetime and set as index
    idx = pd.to_datetime(df[candidate_time_cols[0]], errors="coerce")
    # If some rows failed to parse, drop them together with surplus
    valid = idx.notna()
    surplus_kwh = pd.Series(surplus_col[valid].values, index=idx[valid])
else:
    # No timestamp found — assume hourly series starting at 2024-01-01
    surplus_kwh = pd.Series(
        surplus_col.values,
        index=pd.date_range("2024-01-01", periods=len(surplus_col), freq="H"),
    )

# Optional: ensure strictly hourly frequency (if you have multi-year, it still works).
surplus_kwh = surplus_kwh.sort_index()

# Convert surplus -> H2 with the Pirmasens-Winzeln cap
df_h2 = surplus_to_h2(surplus_kwh, MAX_POWER_KW)
summary = summarize_h2(df_h2, MAX_POWER_KW)

# If your data spans multiple years, aggregate per calendar year as well:
annual_h2 = df_h2["h2_mass_kg"].resample("Y").sum()

# For single-year NPV/LCOH you can use the total period production:
annual_h2_mass = summary["total_h2_mass_kg"]
npv, cfs = npv_h2(annual_h2_mass, MAX_POWER_KW)
pby = payback_year(cfs)
lcoh = lcoh_simple(annual_h2_mass, MAX_POWER_KW)

# Print key results
print("=== Hydrogen from PS_surplus with 1.8 MW cap ===")
print(f"Total surplus electricity: {summary['total_surplus_kwh']:.0f} kWh")
print(f"Usable electricity (electrolyser): {summary['total_usable_kwh']:.0f} kWh")
print(f"Curtailed surplus: {summary['total_curtailed_kwh']:.0f} kWh "
      f"({summary['curtailment_ratio']*100:.1f}%)")
print(f"Hydrogen production (period total): {annual_h2_mass/1000:.2f} tonnes")
print(f"Equivalent full-load hours: {summary['full_load_hours']:.0f} h")

print(f"NPV: €{npv:,.0f}")
print(f"Discounted payback year: {pby}")
print(f"LCOH (approx): €{lcoh:.2f}/kg")

# Optional: also show per-year hydrogen if multi-year data
if len(annual_h2) > 1 or annual_h2.index[0].year != annual_h2.index[-1].year:
    print("\nHydrogen per calendar year (tonnes):")
    print((annual_h2 / 1000).round(2))


=== Hydrogen from PS_surplus with 1.8 MW cap ===
Total surplus electricity: 679620151 kWh
Usable electricity (electrolyser): 3872353 kWh
Curtailed surplus: 675747798 kWh (99.4%)
Hydrogen production (period total): 63.88 tonnes
Equivalent full-load hours: 2151 h
NPV: €7,052,000
Discounted payback year: 5.0
LCOH (approx): €2.02/kg


In [11]:
# Pull the surplus column you mentioned
if "PS_surplus" not in df.columns:
    raise KeyError("Column 'PS_surplus' not found in offset_results.csv")

surplus_col = df["KL+PS_surplus"]

# Build a DatetimeIndex if your CSV has a time column; otherwise synthesize one.
# Try to detect a timestamp-like column.
candidate_time_cols = [c for c in df.columns if "time" in c.lower() or "date" in c.lower()]
if candidate_time_cols:
    # Use the first candidate; parse to datetime and set as index
    idx = pd.to_datetime(df[candidate_time_cols[0]], errors="coerce")
    # If some rows failed to parse, drop them together with surplus
    valid = idx.notna()
    surplus_kwh = pd.Series(surplus_col[valid].values, index=idx[valid])
else:
    # No timestamp found — assume hourly series starting at 2024-01-01
    surplus_kwh = pd.Series(
        surplus_col.values,
        index=pd.date_range("2024-01-01", periods=len(surplus_col), freq="H"),
    )

# Optional: ensure strictly hourly frequency (if you have multi-year, it still works).
surplus_kwh = surplus_kwh.sort_index()

# Convert surplus -> H2 with the Pirmasens-Winzeln cap
df_h2 = surplus_to_h2(surplus_kwh, MAX_POWER_KW)
summary = summarize_h2(df_h2, MAX_POWER_KW)

# If your data spans multiple years, aggregate per calendar year as well:
annual_h2 = df_h2["h2_mass_kg"].resample("Y").sum()

# For single-year NPV/LCOH you can use the total period production:
annual_h2_mass = summary["total_h2_mass_kg"]
npv, cfs = npv_h2(annual_h2_mass, MAX_POWER_KW)
pby = payback_year(cfs)
lcoh = lcoh_simple(annual_h2_mass, MAX_POWER_KW)

# Print key results
print("=== Hydrogen from PS_surplus with 1.8 MW cap ===")
print(f"Total surplus electricity: {summary['total_surplus_kwh']:.0f} kWh")
print(f"Usable electricity (electrolyser): {summary['total_usable_kwh']:.0f} kWh")
print(f"Curtailed surplus: {summary['total_curtailed_kwh']:.0f} kWh "
      f"({summary['curtailment_ratio']*100:.1f}%)")
print(f"Hydrogen production (period total): {annual_h2_mass/1000:.2f} tonnes")
print(f"Equivalent full-load hours: {summary['full_load_hours']:.0f} h")

print(f"NPV: €{npv:,.0f}")
print(f"Discounted payback year: {pby}")
print(f"LCOH (approx): €{lcoh:.2f}/kg")

# Optional: also show per-year hydrogen if multi-year data
if len(annual_h2) > 1 or annual_h2.index[0].year != annual_h2.index[-1].year:
    print("\nHydrogen per calendar year (tonnes):")
    print((annual_h2 / 1000).round(2))

=== Hydrogen from PS_surplus with 1.8 MW cap ===
Total surplus electricity: 2153816560 kWh
Usable electricity (electrolyser): 5656116 kWh
Curtailed surplus: 2148160444 kWh (99.7%)
Hydrogen production (period total): 93.31 tonnes
Equivalent full-load hours: 3142 h
NPV: €11,296,412
Discounted payback year: 4.0
LCOH (approx): €1.39/kg


In [12]:
summary

{'total_surplus_kwh': 2153816559.7319345,
 'total_usable_kwh': 5656115.8502728995,
 'total_curtailed_kwh': 2148160443.881662,
 'curtailment_ratio': 0.9973739101295718,
 'total_h2_energy_kwh': 3676475.3026773855,
 'total_h2_mass_kg': 93311.5559055174,
 'full_load_hours': 3142.286583484944}